In [ ]:
import pandas as pd

# === 1. Carregar os arquivos XLS ===
# ajuste os nomes dos arquivos para os que você baixou do IBGE
renda_path = "dados_renda_00.xlsx"
pop_path = "dados_populacao_00.xlsx"

df_renda = pd.read_excel(renda_path)
df_pop = pd.read_excel(pop_path)

# === 2. Tratar o dataset de renda ===
# Mantemos apenas distrito, total e faixas
df_renda = df_renda.rename(columns={
    df_renda.columns[0]: "bairro",
    "Total (1)": "domicilios_total"
})

# Criar coluna de % de domicílios até 2 SM
cols_ate2 = ["Até 1/2", "Mais de 1/2 a 1", "Mais de 1 a 2"]
df_renda["perc_domicilios_ate2sm"] = (
    df_renda[cols_ate2].sum(axis=1) / df_renda["domicilios_total"] * 100
)

# Criar uma renda média aproximada (usando pesos simples nas classes)
pesos = {
    "Até 1/2": 0.5,
    "Mais de 1/2 a 1": 1,
    "Mais de 1 a 2": 2,
    "Mais de 2 a 5": 3.5,
    "Mais de 5 a 10": 7.5,
    "Mais de 10 a 20": 15,
    "Mais de 20": 25,
    "Sem rendimento (3)": 0
}

def calcular_renda_media(row):
    total = row["domicilios_total"]
    if total == 0:
        return 0
    soma = 0
    for col, peso in pesos.items():
        if col in row:
            soma += row[col] * peso
    return soma / total

df_renda["renda_media_aprox"] = df_renda.apply(calcular_renda_media, axis=1)

# Selecionar apenas colunas relevantes
df_renda = df_renda[["bairro", "domicilios_total", "perc_domicilios_ate2sm", "renda_media_aprox"]]

# === 3. Tratar o dataset de população ===
df_pop = df_pop.rename(columns={
    df_pop.columns[0]: "bairro",
    "2022": "populacao_2022"
})
df_pop = df_pop[["bairro", "populacao_2022"]]

# === 4. Juntar os dois datasets ===
df_final = pd.merge(df_renda, df_pop, on="bairro", how="inner")

# === 5. Exportar CSV consolidado ===
df_final.to_csv("bairros_info.csv", index=False, encoding="utf-8-sig")

print("CSV consolidado gerado com sucesso: bairros_info.csv")


KeyError: "None of [Index(['Até 1/2', 'Mais de 1/2 a 1', 'Mais de 1 a 2'], dtype='object')] are in the [columns]"